# 13 -- Inference Demo (Proof of Deployability)

**Pichle steps ka recap:** `12` mein humne single classifier save kiya
(`health_classifier.joblib` + `feature_config.json` + `winsorization_bounds.json`).
Target ab sirf ek hai: **agle quarter company Healthy hogi ya At-Risk**
(regression aur clustering dono is project se hata diye gaye hain).

Lekin sirf "save hona" kaafi nahi -- **genuinely reusable honi chahiye.**

**Is notebook mein:**
1. Saved classifier + feature-config load karna
2. **CRITICAL BUG jo pehle sandbox testing mein khud pakड़ा gaya tha:** chhote
   sample (ya sirf 1 nayi company) par prediction karne ki koshish karne se
   sector one-hot-encoding columns MISALIGN ho jate hain
3. Fix (`EXPECTED_COLS`-driven `reindex`) + verified working demo, real
   **test-set** companies par (jin ka data model ne training ke waqt kabhi
   nahi dekha)

## Working directory fix (zaroori)

In [1]:
import os
from pathlib import Path

if not Path("data").exists():
    os.chdir("..")

print("Working directory:", os.getcwd())
assert Path("data/processed").exists(), "data/processed nahi mila -- path check karo"

Working directory: d:\Developing_Projects\cross-industry-financial-analysis_Project


## Setup + Load Artifacts

In [2]:
import pandas as pd
import numpy as np
import joblib
import json
from pathlib import Path

PROCESSED_DIR = Path("data/processed")
MODEL_DIR = Path("models")

clf = joblib.load(MODEL_DIR / "health_classifier.joblib")

with open(MODEL_DIR / "feature_config.json") as f:
    config = json.load(f)

EXPECTED_COLS = config["feature_columns"]
RATIO_GROWTH_FEATURES = [c for c in config["features_numeric_binary"]
                          if c not in ("is_annual_filing", "has_inventory")]
CONTEXT_FEATURES = ["is_annual_filing", "has_inventory"]
CLASSES = config["classes"]

print("Classifier + feature-config load ho gaye.")
print(f"Target: {config['target']}")
print(f"Expected feature columns: {len(EXPECTED_COLS)}")
print(f"Classes: {CLASSES}")

Classifier + feature-config load ho gaye.
Target: target_next_quarter_health
Expected feature columns: 22
Classes: {'0': 'At-Risk', '1': 'Healthy'}


---
# CRITICAL BUG (sandbox testing mein khud pakड़ा gaya)

**Kya hua:** Jab hum sirf 2-3 (ya kam) NAYI companies par predict karne ki
koshish karte hain, `pd.get_dummies(industry_sector)` sirf UN sectors ke
liye columns banata hai jo us CHHOTE sample mein maujood hain -- baaki
sector-columns (jo model training ke waqt expect karta hai) MISSING ho
jate hain. Isse prediction crash ho jati ya features galat align ho jati
(silently -- koi error diye bina).

Neeche PEHLE demonstrate karte hain BUG (bina fix), phir FIX.

In [3]:
df = pd.read_parquet(PROCESSED_DIR / "10_final_dataset.parquet")

with open(PROCESSED_DIR / "company_split.json") as f:
    split = json.load(f)
test_ciks = set(split["test_ciks"])

# Demo ke liye TEST companies mein se select karte hain -- model ne inhein
# training ke waqt kabhi nahi dekha, isliye ye asal "naye data par prediction"
# ka honest simulation hai.
test_rows = df[df["cik"].isin(test_ciks) & df["target_next_quarter_health"].notna()]
sample = test_rows.sample(3, random_state=7)
print("Sample companies:", sample["name"].tolist())
print("Sample industries:", sample["industry_sector"].tolist())

# BUG demonstration: naive feature-building (WITHOUT fix)
industry_dummies_naive = pd.get_dummies(sample["industry_sector"], prefix="sector")
X_naive = pd.concat([sample[RATIO_GROWTH_FEATURES+CONTEXT_FEATURES].reset_index(drop=True),
                     industry_dummies_naive.reset_index(drop=True)], axis=1)

print(f"\nNaive feature matrix shape: {X_naive.shape} (model expects {len(EXPECTED_COLS)} columns)")
missing = set(EXPECTED_COLS) - set(X_naive.columns)
print(f"MISSING columns (bug confirmed): {sorted(missing)}")

Sample companies: ['KENVUE INC.', 'NEXTPLAT CORP', 'FISERV INC']
Sample industries: ['Manufacturing', 'Transport_Utilities', 'Services']

Naive feature matrix shape: (3, 16) (model expects 22 columns)
MISSING columns (bug confirmed): ['sector_Agriculture', 'sector_Construction', 'sector_Finance_RealEstate', 'sector_Mining', 'sector_Retail', 'sector_Wholesale']


**Business Insight:**

**Ye exactly wahi tarah ka bug hai jo real-world deployment mein sabse zyada
common hai** -- training-time aur inference-time feature-engineering
"silently" mismatch ho jati hai. Agar isay pakड़a na jata, model ya to CRASH
hota, ya (zyada khatarnak) **galat columns ko galat jagah predict kar deta**
bina koi error diye.

## FIX -- Reindex to exact training-time columns

In [4]:
def build_inference_features(data: pd.DataFrame) -> pd.DataFrame:
    """
    Naye/kam-sample data ke liye SAHI feature-matrix banata hai --
    training-time ke EXACT columns (order + count) ke sath, missing
    dummy-columns ko 0 se fill karke.
    """
    industry_dummies = pd.get_dummies(data["industry_sector"], prefix="sector")
    X = pd.concat([
        data[RATIO_GROWTH_FEATURES + CONTEXT_FEATURES].reset_index(drop=True),
        industry_dummies.reset_index(drop=True),
    ], axis=1)
    X = X.reindex(columns=EXPECTED_COLS, fill_value=0)
    return X

X_fixed = build_inference_features(sample)
print(f"Fixed shape: {X_fixed.shape}")
print(f"Columns exactly match training: {list(X_fixed.columns) == EXPECTED_COLS}")
assert list(X_fixed.columns) == EXPECTED_COLS, "MASLA: columns abhi bhi match nahi kar rahe!"
print("[OK] Feature-alignment bug FIXED aur verified.")

Fixed shape: (3, 22)
Columns exactly match training: True
[OK] Feature-alignment bug FIXED aur verified.


---
# Live Demo: 3 Real (Test-Set) Companies par Prediction

In [5]:
health_pred_idx = clf.predict(X_fixed)
health_proba = clf.predict_proba(X_fixed)
healthy_col = list(clf.classes_).index(1)  # 1 = "Healthy" (feature_config.json ke classes dekho)

health_pred_label = np.where(health_pred_idx == 1, "Healthy", "At-Risk")
actual_label = np.where(sample["target_next_quarter_health"].values == 1, "Healthy", "At-Risk")

results = pd.DataFrame({
    "Company": sample["name"].values,
    "Industry": sample["industry_sector"].values,
    "Predicted_Health": health_pred_label,
    "P(Healthy)": health_proba[:, healthy_col].round(3),
    "ACTUAL_Health (ground truth)": actual_label,
})
print(results.to_string(index=False))

      Company            Industry Predicted_Health  P(Healthy) ACTUAL_Health (ground truth)
  KENVUE INC.       Manufacturing          At-Risk       0.132                      At-Risk
NEXTPLAT CORP Transport_Utilities          At-Risk       0.033                      At-Risk
   FISERV INC            Services          Healthy       0.799                      At-Risk


In [6]:
n_matches = (health_pred_label == actual_label).sum()
print(f"\nVerifiable predictions: {n_matches}/{len(sample)} matched ground-truth "
      f"(chhota demo-sample, in companies ka data model ne training mein kabhi nahi dekha)")


Verifiable predictions: 2/3 matched ground-truth (chhota demo-sample, in companies ka data model ne training mein kabhi nahi dekha)


**Business Insight:**

Ye teenon companies **locked test-set** (`company_split.json`) se hain --
matlab classifier ne inka data na training mein, na winsorization bounds
banate waqt, na feature-selection mein dekha tha. Isliye ye demo ek honest
"naya/unseen data par prediction" simulation hai, sirf illustrative sample
nahi.

Poore test-set (981 companies, 7,051 rows) par model ki performance `12`
mein already report ki ja chuki hai (Accuracy 88.0%, ROC-AUC 0.939) -- ye
3-company demo sirf deployment-readiness ka proof hai, statistical claim
nahi.

---
# Sanity Checks

In [7]:
assert X_fixed.shape[1] == len(EXPECTED_COLS)
print("[OK] Feature-alignment fix verified working.")

assert set(health_pred_label).issubset({"Healthy", "At-Risk"})
print("[OK] Predictions valid class-labels de rahi hain.")

assert list(clf.classes_) == [0, 1], f"Class order unexpected: {clf.classes_}"
print("[OK] Model class-order confirmed: 0=At-Risk, 1=Healthy.")

print("\n=== SINGLE-TARGET PROJECT GENUINELY COMPLETE + DEPLOYMENT-VERIFIED ===")

[OK] Feature-alignment fix verified working.
[OK] Predictions valid class-labels de rahi hain.
[OK] Model class-order confirmed: 0=At-Risk, 1=Healthy.

=== SINGLE-TARGET PROJECT GENUINELY COMPLETE + DEPLOYMENT-VERIFIED ===
